## Chroma

In [3]:
##Building a simple vectordb
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
loader = TextLoader("speech.txt")
data = loader.load()
data

[Document(metadata={'source': 'speech.txt'}, page_content='The Duffield Memorial is a gravesite monument located in the churchyard of the Church of St Mary in Great Baddow, England. Designed by Herbert Maryon and installed in 1912, it commemorates Marianne and W. W. Duffield, who died in 1910 and 1912, respectively. A second plaque commemorates their son, W. B. Duffield, who died in 1918. The memorial is made of riveted sections of bronze sheet metal and comprises edging and a vertical cross. The edging follows the rectangular perimeter of the grave plot, with short pillars at each corner. Within the plot sits a Celtic wheel cross, decorated in relief with leaflike motifs. A curved shaft connects it to the foot; both the shaft and the four-sided base upon which it is mounted have curved and splayed sides. The plaques commemorating the Duffields are riveted to the base; a medallion, now lost, was once riveted to the centre of the cross. In 2022, Historic England designated the work a Gr

In [5]:
## Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=0)
splits = text_splitter.split_documents(data)

In [7]:
embedding = OllamaEmbeddings(model="nomic-embed-text")
vectordb = Chroma.from_documents(documents = splits,embedding=embedding)
vectordb

In [9]:
## Quey it
query = "what does the speaker believe "
docs = vectordb.similarity_search(query)
docs[0].page_content

'plot, with short pillars at each corner. Within the plot sits a Celtic wheel cross, decorated in relief with leaflike motifs. A curved shaft connects it to the foot; both the shaft and the four-sided base upon which it is mounted have curved and splayed sides. The plaques commemorating the Duffields are riveted to the base; a medallion, now lost, was once riveted to the centre of the cross. In 2022, Historic England designated the work a Grade II listed building, noting its unusual Art Nouveau'

In [10]:
## Saving to the Disk
vectordb = Chroma.from_documents(documents=splits,embedding=embedding,persist_directory="./chroma_db")

In [12]:
# load from disk
db2 = Chroma(persist_directory="./chroma_db",embedding_function=embedding)
docs = db2.similarity_search(query)
print(docs[0].page_content)

plot, with short pillars at each corner. Within the plot sits a Celtic wheel cross, decorated in relief with leaflike motifs. A curved shaft connects it to the foot; both the shaft and the four-sided base upon which it is mounted have curved and splayed sides. The plaques commemorating the Duffields are riveted to the base; a medallion, now lost, was once riveted to the centre of the cross. In 2022, Historic England designated the work a Grade II listed building, noting its unusual Art Nouveau


In [13]:
## similarity search with score
docs = vectordb.similarity_search_with_score(query)
docs

[(Document(id='6309f8b6-2dce-4128-b7ef-0b0224794070', metadata={'source': 'speech.txt'}, page_content='plot, with short pillars at each corner. Within the plot sits a Celtic wheel cross, decorated in relief with leaflike motifs. A curved shaft connects it to the foot; both the shaft and the four-sided base upon which it is mounted have curved and splayed sides. The plaques commemorating the Duffields are riveted to the base; a medallion, now lost, was once riveted to the centre of the cross. In 2022, Historic England designated the work a Grade II listed building, noting its unusual Art Nouveau'),
  509.04754638671875),
 (Document(id='8a4683bf-cb90-4374-805e-db0e793b2568', metadata={'source': 'speech.txt'}, page_content='The Duffield Memorial is a gravesite monument located in the churchyard of the Church of St Mary in Great Baddow, England. Designed by Herbert Maryon and installed in 1912, it commemorates Marianne and W. W. Duffield, who died in 1910 and 1912, respectively. A second p

In [15]:
## Retriever option
retriever = vectordb.as_retriever()
retriever.invoke(query)[0].page_content

'plot, with short pillars at each corner. Within the plot sits a Celtic wheel cross, decorated in relief with leaflike motifs. A curved shaft connects it to the foot; both the shaft and the four-sided base upon which it is mounted have curved and splayed sides. The plaques commemorating the Duffields are riveted to the base; a medallion, now lost, was once riveted to the centre of the cross. In 2022, Historic England designated the work a Grade II listed building, noting its unusual Art Nouveau'